# Stemmekloning i Google Colab — IndexTTS2 Premium (SECourses)

Denne notatblokka installerer **SECourses sitt grensesnitt** til IndexTTS2, ikke det enkle
standardgrensesnittet. Det gir dere forhåndsinnstillinger som kan lagres, opptak rett fra
mikrofonen, avbryt-knapp, og følelsesvalg på engelsk.

**Les dette først:**

1. Klikk **Kjøretid → Endre type kjøretid → Maskinvareakselerator: T4 GPU → Lagre.**
   Uten GPU kommer ingenting til å fungere.
2. Kjør cellene i rekkefølge, ovenfra og ned.
3. Steg 1 og 2 tar til sammen **15–25 minutter**. Start dem med én gang, og gjør opptakene
   mens de kjører.
4. Alt forsvinner når Colab-økta lukkes. **Last ned lydfilene før dere går** (Steg 5).

> Dere kloner bare deres egen stemme. Ingenting publiseres.


## Steg 0 – Sjekk at dere har fått GPU

In [ ]:
!nvidia-smi

import torch
print()
print("GPU tilgjengelig:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Skjermkort:", torch.cuda.get_device_name(0))
else:
    print("STOPP: Kjoretid -> Endre type kjoretid -> T4 GPU, og kjor denne cellen paa nytt.")


## Steg 1 – Hent appen og installer

Vi henter SECourses-appen fra GitHub og bygger et eget Python 3.10-miljø til den.
Colab kjører en nyere Python enn appen tåler, så `uv` laster ned 3.10 for oss.

**Tar 8–15 minutter.** Mange linjer med tekst er normalt.

In [ ]:
%cd /content
!git clone https://github.com/FurkanGozukara/Premium_IndexTTS2_SECourses
%cd /content/Premium_IndexTTS2_SECourses
!git pull

!pip install -q -U uv
print("\nuv installert.")


Appen leveres normalt med en `requirements.txt` i zip-fila fra Patreon. Den ligger ikke i
GitHub-repoet, så vi skriver en Colab-tilpasset versjon her.

**To pakker er med vilje fjernet:** `flash_attn` og `sageattention`. De gjør appen raskere,
men krever et nyere skjermkort enn T4-en Colab gir bort gratis. Prøver man å bruke dem på en
T4, krasjer appen.

In [ ]:
requirements = """--extra-index-url https://download.pytorch.org/whl/cu129
torch==2.8.0
torchvision
torchaudio==2.8.0

accelerate==1.8.1
cn2an==0.5.22
cython==3.0.7
descript-audiotools==0.7.2
einops>=0.8.1
ffmpeg-python==0.2.0
g2p-en==2.1.0
jieba==0.42.1
json5==0.10.0
keras==2.9.0
librosa==0.10.2.post1
matplotlib==3.8.2
modelscope==1.27.0
munch==4.0.0
numba==0.58.1
numpy==1.26.2
omegaconf>=2.3.0
opencv-python==4.9.0.80
pandas==2.3.2
safetensors==0.5.2
sentencepiece>=0.2.1
tensorboard==2.9.1
textstat>=0.7.10
tokenizers==0.21.0
tqdm>=4.67.1
transformers==4.52.1
hf_xet
pydub
WeTextProcessing
gradio==6.11.0
"""

with open("/content/requirements_colab.txt", "w") as f:
    f.write(requirements)

print("Skrev /content/requirements_colab.txt")


In [ ]:
import os

os.environ["UV_SKIP_WHEEL_FILENAME_CHECK"] = "1"
os.environ["UV_LINK_MODE"] = "copy"

# Eget Python 3.10-miljo, isolert fra Colab sitt eget
!uv venv --python 3.10 /content/venv310

!uv pip install --python /content/venv310/bin/python -r /content/requirements_colab.txt --index-strategy unsafe-best-match

print("\nSteg 1 ferdig.")
!/content/venv310/bin/python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"


## Steg 2 – Last ned modellene

Dette gjør det samme som `HF_model_downloader.py` i Patreon-pakka: henter hovedmodellen og
fire hjelpemodeller.

**Tar 5–10 minutter.** Til sammen noen gigabyte.

In [ ]:
import os, shutil, glob

!pip install -q -U "huggingface_hub[hf_xet]"
from huggingface_hub import snapshot_download, hf_hub_download

APP  = "/content/Premium_IndexTTS2_SECourses"
CKPT = os.path.join(APP, "checkpoints")
CACHE = os.path.join(CKPT, "hf_cache")
os.makedirs(CKPT, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --- 1. Hovedmodellen -------------------------------------------------
print("Laster ned hovedmodellen ...")
tmp = "/content/_dl_tmp"
snapshot_download(
    repo_id="MonsterMMORPG/Wan_GGUF",
    allow_patterns=["Index_TTS2/*"],
    local_dir=tmp,
)

# Filene skal ligge rett i checkpoints/, ikke i en Index_TTS2-undermappe
kilde = os.path.join(tmp, "Index_TTS2")
for navn in os.listdir(kilde):
    maal = os.path.join(CKPT, navn)
    if os.path.exists(maal):
        shutil.rmtree(maal) if os.path.isdir(maal) else os.remove(maal)
    shutil.move(os.path.join(kilde, navn), maal)
shutil.rmtree(tmp, ignore_errors=True)

# --- 2. Hjelpemodellene ----------------------------------------------
hjelpemodeller = [
    ("facebook/w2v-bert-2.0", ["config.json", "model.safetensors", "preprocessor_config.json"]),
    ("amphion/MaskGCT", ["semantic_codec/model.safetensors"]),
    ("funasr/campplus", ["campplus_cn_common.bin"]),
    ("nvidia/bigvgan_v2_22khz_80band_256x", ["bigvgan_generator.pt", "config.json"]),
]

for repo, filer in hjelpemodeller:
    print(f"\nLaster ned {repo} ...")
    for fil in filer:
        hf_hub_download(repo_id=repo, filename=fil, repo_type="model", cache_dir=CACHE)

# --- 3. Kontroll ------------------------------------------------------
print("\n" + "=" * 55)
mangler = [f for f in ["bpe.model", "gpt.pth", "config.yaml", "s2mel.pth",
                       "wav2vec2bert_stats.pt"]
           if not os.path.exists(os.path.join(CKPT, f))]
if mangler:
    print("MANGLER filer:", mangler, "- kjor cella paa nytt")
else:
    print("Alle nodvendige modellfiler er paa plass.")
print("=" * 55)


## Steg 3 – Start appen

SECourses-appen har en innebygd delingsfunksjon (`--share`), så den lager selv en offentlig
lenke som slutter på `gradio.live`. Alle i gruppa kan åpne den samtidig.

Første oppstart tar **2–5 minutter**. Vent til dere ser `>>> ÅPEN LENKE`.

In [ ]:
# ---------------------------------------------------------------
FP16 = True     # halv presisjon: raskere og mindre minnebruk
# ---------------------------------------------------------------

import os, re, socket, subprocess, time

APP  = "/content/Premium_IndexTTS2_SECourses"
CKPT = os.path.join(APP, "checkpoints")

miljo = dict(os.environ)
miljo["PYTHONWARNINGS"]  = "ignore"
miljo["HF_HOME"]         = CKPT                              # som i Windows_Start_App.bat
miljo["HF_HUB_CACHE"]    = os.path.join(CKPT, "hf_cache")    # der Steg 2 la hjelpemodellene
miljo["GRADIO_ANALYTICS_ENABLED"] = "False"

args = ["/content/venv310/bin/python", "webui.py",
        "--model_dir", CKPT,
        "--host", "0.0.0.0", "--port", "7860",
        "--share"]
if FP16:
    args.append("--fp16")

logg_sti = "/content/webui.log"
logg = open(logg_sti, "w")
app = subprocess.Popen(args, cwd=APP, env=miljo, stdout=logg, stderr=subprocess.STDOUT)
print("Starter IndexTTS2 Premium ... dette tar 2-5 minutter.")


def port_apen(port=7860):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0


url = None
for i in range(150):                      # maks ca. 12 minutter
    tekst = open(logg_sti).read()
    treff = re.search(r"https://[-\w]+\.gradio\.live", tekst)
    if treff:
        url = treff.group(0)
        break
    if app.poll() is not None:
        print("\n!! Appen krasjet. Siste del av loggen:\n")
        print(tekst[-3000:])
        break
    if i % 12 == 0 and i > 0:
        print(f"   ... venter fortsatt ({i*5} sekunder)")
    time.sleep(5)

if url:
    print("\n" + "=" * 60)
    print(">>> APEN LENKE (apne i ny fane, del med gruppa):")
    print("   ", url)
    print("=" * 60)
elif app.poll() is None:
    print("\nFant ingen delingslenke, men appen kjorer kanskje likevel.")
    if port_apen():
        from google.colab.output import eval_js
        print("Bruk denne i stedet (virker bare i DIN nettleser):")
        print(eval_js("google.colab.kernel.proxyPort(7860)"))
    else:
        print("Siste del av loggen:\n")
        print(open(logg_sti).read()[-3000:])


### Hvis noe går galt

| Det som skjer | Hva dere gjør |
|---|---|
| `Model directory ... does not exist` | Steg 2 ble ikke ferdig. Kjør den om igjen |
| `Required file ... does not exist` | Samme: kjør Steg 2 om igjen |
| `CUDA out of memory` | Kortere tekst, og sjekk at `FP16 = True` |
| `FlashAttention only supports Ampere` | En pakke har sneket seg inn. Kjør Steg 1 om igjen |
| Ingenting skjer på 12 minutter | Kjør cella på nytt. Hjelper ikke det: Kjøretid → Koble fra og slett kjøretid, start fra Steg 0 |
| Lenken sier «Bad Gateway» | Vent ett minutt til og oppdater |

Trenger dere å se hva som skjer under panseret, kjør cella under.

In [ ]:
print(open("/content/webui.log").read()[-4000:])


## Steg 4 – Slik bruker dere grensesnittet

Dette er SECourses-versjonen, så den ser annerledes ut enn standard IndexTTS.

**Referansestemme:** last opp lydfila, eller ta opp rett fra mikrofonen i nettleseren
(anbefalt lengde 15 sekunder). Appen godtar også videofiler og trekker ut lyden selv.

**Emotion control** har fire valg:

| Valg | Hva det gjør |
|---|---|
| *Same as speaker voice* | Følelsen hentes fra referanseopptaket |
| *Use emotion reference audio* | Et **eget** klipp styrer bare følelsen |
| *Use emotion vector control* | Åtte skyveknapper: glad, sint, trist, redd, avsky, tungsindig, overrasket, rolig |
| *Use emotion text description* | Beskriv følelsen med ord på engelsk, f.eks. `whispering, secretive` |

**Nyttige knapper i denne versjonen:**

- **Save / Load preset** — lagre innstillingene deres og hent dem tilbake. Bruk dette. Da kan
  dere sammenligne to oppsett uten å skrive alt om igjen.
- **Cancel** — stopp en generering som har låst seg
- **Max tokens per segment** — lange tekster deles opp automatisk
- Ferdige filer nummereres og legges i `outputs/`

Ett råd: **endre én ting av gangen**, og lagre en preset når dere finner noe som funker.

## Steg 5 – Last ned lydfilene

Kjør denne **før dere lukker Colab.** Alt forsvinner ellers.

In [ ]:
import shutil, os
from google.colab import files

UT = "/content/Premium_IndexTTS2_SECourses/outputs"
os.makedirs(UT, exist_ok=True)

print("Filer i outputs/:")
!ls -R "$UT" | head -50

shutil.make_archive("/content/lydfiler", "zip", UT)
files.download("/content/lydfiler.zip")


---

## Bonus – Ordentlig norsk uttale med Chatterbox

IndexTTS2 kan kinesisk, engelsk og japansk. **Norsk står ikke på lista**, så norsk tekst blir
uttalt med engelske uttaleregler.

Chatterbox Multilingual fra Resemble AI støtter norsk (`no`) direkte. Den har mindre
finkornet følelseskontroll, men mye bedre norsk uttale.

**Viktig:** kjør denne delen i en **ny, tom Colab-notatblokk**, ikke i samme kjøretid som
IndexTTS. De to installasjonene kan krangle om versjoner.

Chatterbox legger inn et usynlig **vannmerke** i all lyd den lager, slik at generert tale skal
kunne spores. Snakk om hvorfor i gruppa.

In [ ]:
!pip install -q chatterbox-tts
print("Ferdig. Kjor neste celle.")


In [ ]:
import torch
import gradio as gr

try:
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS as CB
    flerspraaklig = True
except ImportError:
    from chatterbox.tts import ChatterboxTTS as CB
    flerspraaklig = False

enhet = "cuda" if torch.cuda.is_available() else "cpu"
modell = CB.from_pretrained(device=enhet)
print("Modell lastet paa", enhet, "| flerspraaklig:", flerspraaklig)


def lag_tale(referanse, tekst, uttrykk, stabilitet):
    if referanse is None:
        raise gr.Error("Last opp et referanseopptak forst.")
    try:
        wav = modell.generate(tekst, language_id="no", audio_prompt_path=referanse,
                              exaggeration=uttrykk, cfg_weight=stabilitet)
    except TypeError:
        wav = modell.generate(tekst, audio_prompt_path=referanse,
                              exaggeration=uttrykk, cfg_weight=stabilitet)
    return (modell.sr, wav.squeeze(0).cpu().numpy())


gr.Interface(
    fn=lag_tale,
    inputs=[
        gr.Audio(type="filepath", label="Referansestemme (15-25 sekunder)"),
        gr.Textbox(label="Tekst paa norsk", lines=4,
                   value="Morna, Jens. Takk for alt du har gjort for oss."),
        gr.Slider(0.2, 1.5, value=0.6, step=0.05,
                  label="Uttrykk (0.5 = nokternt, 1.0+ = dramatisk)"),
        gr.Slider(0.1, 1.0, value=0.4, step=0.05,
                  label="Stabilitet (lavere = mer variasjon i tempo)"),
    ],
    outputs=gr.Audio(label="Klonet stemme"),
    title="Stemmekloning paa norsk",
    description="Bruk bare opptak du har samtykke til a bruke.",
).launch(share=True)
